In [ ]:
import requests
import json
import logging
from config import Config
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import time
import re

# Configure root logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)


def identify_plant(data, files):
    api_endpoint = f"{Config.PLANTNET_API}{Config.PROJECT}?api-key={Config.PLANTNET_KEY}"
    req = requests.Request('POST', url=api_endpoint, files=files, data=data)
    prepared = req.prepare()

    s = requests.Session()
    response = s.send(prepared)
    json_result = json.loads(response.text)
    results = json_result["results"]

    score = results[0]["score"]
    common_names = results[0]["species"]["commonNames"]
    scientific_name = results[0]["species"]["scientificName"]

    return {"score": score, "common_names": common_names, "scientific_name": scientific_name}


def build_dict(arr):
    result = {}
    for item in arr:
        item_id = item.get('id', 'unknown')
        try:
            # Validate required keys
            t = item['type']
            desc = item['description']
            
            # Ensure types are correct
            if not isinstance(t, str) or not isinstance(desc, str):
                raise TypeError(f"Invalid data types for 'type' or 'description' in item with id {item.get('id')}")
            
            # Check for duplicates
            if t in result:
                raise ValueError(f"Duplicate entry for type '{t}' found in item with id {item.get('id')}")
            
            result[t] = desc
            logger.info("Added entry for type '%s' (id=%s)", t, item_id)
        
        except KeyError as ke:
            logger.error("Missing key %r in item %s", ke, item_id)
        except (TypeError, ValueError) as ve:
            logger.error("Error processing item %s: %s", item_id, ve)
        except Exception as e:
            logger.exception("Unexpected error processing item %s", item_id)
    logger.info("Finished building dict; %d entries added", len(result))
    return result


def get_care_guide(plant_name):
    """Fetch the care guide for a given plant name.

    This helper wraps the Perenual API call with some basic error handling so
    callers don't have to deal with network or parsing errors.
    """
    
    url = (
        f"{Config.PERENUAL_API}{Config.PERENUAL_ENDPOINT}?key={Config.PERENUAL_KEY}&q={plant_name}"
    )

    try:
        response = requests.request("GET", url, headers={}, data={})
        response.raise_for_status()
    except requests.RequestException as exc:
        logger.error("Error requesting care guide for %s: %s", plant_name, exc)
        return {}

    try:
        json_data = json.loads(response.text)
    except json.JSONDecodeError as exc:
        logger.error("Invalid JSON received for %s: %s", plant_name, exc)
        return {}

    try:
        top_result = json_data["data"][0]
        care_dict = build_dict(top_result["section"])
    except (KeyError, IndexError, TypeError) as exc:
        logger.error("Unexpected response structure for %s: %s", plant_name, exc)
        return {}

    return care_dict


def search_perenual_website(query, delay=1):
    base_url = "https://perenual.com/plant-species-database-search-finder"
    params = {"search": query}

    # 2. Issue the GET request
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; MyPlantBot/1.0; +https://yourdomain.com/bot)"
    }
    resp = requests.get(base_url, params=params, headers=headers)
    resp.raise_for_status()  # will raise an error for HTTP codes ≥400

    soup = BeautifulSoup(resp.text, "html.parser")
    urls = set()
    # find all <a> tags whose href begins with the species path
    for a in soup.select('.search-container-box'):
        full = urljoin(base_url, a["href"])
        urls.add(full)

    time.sleep(delay)
    return list(urls)


def get_specific_perennial_page(url):
    # print("Getting specific perennial page")
    resp = requests.get(url)
    resp.raise_for_status()
    # print("Fetched perennial page")
    # print("Parsing HTML")
    soup = BeautifulSoup(resp.text, "html.parser")
    # print("Parsed HTML, selecting for care descriptions")
    paras = soup.select("p.whitespace-pre-wrap")
    # print("Getting text specifically")
    texts = [p.get_text(strip=True) for p in paras]
    # print("Formatting care descriptions")    
    result = {
        "watering":texts[0],
        "sunlight":texts[1],
        "pruning":texts[2]
    }
    return result


def get_plant_info(image_files, organs=None):
    files = []
    for image_file in image_files:
        image_data = open(image_file, 'rb')
        files.append(('images', image_data))

    if organs is None:
        organs = ["auto" for _ in image_files]

    data = {'organs': organs}

    identified_plant_result = identify_plant(data, files)
    
    care_dict = get_care_guide(identified_plant_result['scientific_name'])
    if care_dict == {}:
        for name in identified_plant_result['common_names']:
            care_dict = get_care_guide(name)
            if care_dict != {}:
                break
        print(care_dict)
        if care_dict == {}:
            for name in identified_plant_result['common_names']:
                description_urls = search_perenual_website(name)
                if len(description_urls) > 0:
                    break
            care_dict = get_specific_perennial_page(description_urls[0])

    identified_plant_result.update(care_dict)
    for key, value in identified_plant_result.items():
        print(f"\n{key.upper()}:")
        print(value)

    return identified_plant_result




In [32]:
results_mint = get_plant_info("./test_photos/mint.jpg")

2025-07-22 15:31:03 [ERROR] __main__: Unexpected response structure for Mentha spicata L.: list index out of range



SCORE:
0.12729

COMMON_NAMES:
['Spearmint', 'Wild mint', 'Garden mint']

SCIENTIFIC_NAME:
Mentha spicata L.

WATERING:
Spearmint should be watered frequently, but only lightly. It should be kept consistently moist but not wet. It is best to water early in the day so that the leaves have time to dry before nightfall. During the summer months, the plant may need to be watered more frequently. In the winter, water lightly about once a week. Overwatering could lead to root rot so be sure to let the soil dry out between waterings.

SUNLIGHT:
Spearmint (Mentha spicata 'Kentucky Colonel') is a perennial herb that thrives in full sun and a minimum of 6 hours of direct sunlight a day. It should be planted in a sunny location with partial shade, for example, during the afternoon when temperatures are hotter. If planted in a partially shaded area, it may take away from the flavor of the spearmint because it may not have enough hours of sunlight to blossom. As a general rule, the more hours of di

In [45]:
results_rockrose = get_plant_info("./test_photos/rockrose.jpg")

2025-07-22 15:39:07 [ERROR] __main__: Unexpected response structure for Cistus crispus L.: list index out of range
2025-07-22 15:39:08 [ERROR] __main__: Unexpected response structure for Wrinkle-leaved rockrose: list index out of range
2025-07-22 15:39:09 [ERROR] __main__: Unexpected response structure for Rock rose: list index out of range
2025-07-22 15:39:10 [ERROR] __main__: Unexpected response structure for Wavy-leaf Rock Rose: list index out of range


{}

SCORE:
0.95973

COMMON_NAMES:
['Wrinkle-leaved rockrose', 'Rock rose', 'Wavy-leaf Rock Rose']

SCIENTIFIC_NAME:
Cistus crispus L.

WATERING:
Rock Rose should be watered well and deeply, about once a week. During hot, dry spells, water more often, making sure the soil is moist but not saturated. During cold spells, reduce watering, allowing the soil to partially dry between watering.

Re-potting or fertilizing should be done annually in late winter or early spring. With the right care, Rock Rose will continue to grow and flower beautifully.

SUNLIGHT:
Rock roses prefer full sun, but can tolerate light shade. They should receive at least 6 hours of direct sunlight a day, ideally between the hours of 8am and 4pm. Provide full sun in the spring and summer, but allow some afternoon shade in the hottest months of the year to prevent too much heat stress on the plant. Avoid exposing your rock rose to full sun during the hottest part of the day, especially in the summer. In the fall and wi

In [34]:
results_lavender = get_plant_info("./test_photos/lavender.jpg")

2025-07-22 15:31:48 [ERROR] __main__: Unexpected response structure for Lavandula angustifolia Mill.: list index out of range



SCORE:
0.73563

COMMON_NAMES:
['English lavender', 'Common Lavender', 'Lavender']

SCIENTIFIC_NAME:
Lavandula angustifolia Mill.

WATERING:
English lavender prefers well-drained soil and full sun. Water your English lavender regularly, once every 5 to 7 days if there has been no rain. During the summertime, you may need to water twice per week, especially if plants are in containers or in very exposed areas. Generally, you should give your lavenders a good soak until moisture runs through the drainage holes of the pot and the water on the surface starts to evaporate. However, avoid overwatering the lavenders, as this can cause root rot. You should reduce your watering during the winter months, so the soil is only damp but not soggy.

SUNLIGHT:
English lavender (Lavandula angustifolia 'Lavandula Schola' BLUE CUSHION) plants enjoy full sun, providing them with 6 to 8 hours of direct sunlight each day. They can tolerate some light shade, however this will likely reduce flowering. When pl

In [35]:
results_camellia = get_plant_info("./test_photos/camellia.jpg")

2025-07-22 15:32:30 [ERROR] __main__: Unexpected response structure for Camellia sasanqua Thunb.: list index out of range



SCORE:
0.36081

COMMON_NAMES:
['Camellia', 'Sasanqua camellia', 'Christmas camellia']

SCIENTIFIC_NAME:
Camellia sasanqua Thunb.

WATERING:
Camellia sasanqua requires regular watering during the summer months. Generally, this plant should be watered every 1 to 2 weeks for optimal growth. In warmer climates, it may need extra water during the warmer months, but it should not be watered every day.

In cooler temperatures, Camellia sasanqua should be watered once every 2 to 3 weeks. During this time, the soil should be kept moist, but not soggy. If the soil is dry approximately 1 inch below the surface, it’s time to water.

In periods of extreme heat, Camellia sasanqua needs to be watered more frequently to ensure proper hydration and to prevent burning of the foliage. During these times, you should water the plant every week to ensure proper hydration.

In the winter, Camellia sasanqua does not need as much water, and can even tolerate some dry conditions. The soil should be kept on the